# 01 · 搭建一个小型灰度手势 CNN

本章先不依赖数据集。我们用一个占位图像张量理解：输入形状 → 特征提取 → 类别分数 → 损失与梯度。
任务是固定 ROI 的石头、剪刀、布分类；这不是检测器，也尚未训练好。

## 1. 导入最少的组件

模型定义放在独立 Python 文件，notebook 负责实验和解释；后续训练与导出复用同一份源码。

In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "cnn_tutorial").is_dir() and (p / "MAIN.md").exists()), None)
if ROOT is None:
    raise RuntimeError("请从 CNN-Tutorial 项目目录打开此 notebook")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project:", ROOT.name)

import torch
from torch import nn
from cnn_tutorial.model import GrayGestureCNN, profile_model
from cnn_tutorial.training import seed_everything
seed_everything(42)

Project: CNN-Tutorial


## 2. 一张灰度图怎样表示

PyTorch 卷积输入采用 NCHW：样本数、通道数、高、宽。一张 64×64 灰度图的输入是 `(1,1,64,64)`。
这里用随机数只检查接口；它不用于训练成绩，也不能作为真实 INT8 校准数据。

In [2]:
x = torch.rand(1, 1, 64, 64)
print("shape:", tuple(x.shape), "dtype:", x.dtype)
print("range:", float(x.min()), float(x.max()))

shape: (1, 1, 64, 64) dtype: torch.float32
range: 0.0011727213859558105 0.9998044371604919


## 3. 阅读模型定义

卷积学习局部模式，ReLU 引入非线性，池化缩小空间尺寸。通道数逐步从 1→8→16→32。
末尾保留 4×4 网格，线性层将 512 个特征映射为三个分数。
下面直接显示共享源码，修改结构时请编辑 `cnn_tutorial/model.py`。

In [3]:
import inspect
print(inspect.getsource(GrayGestureCNN))
model = GrayGestureCNN()
print(model)

class GrayGestureCNN(nn.Module):
    """Input NCHW float32 in [0, 1], output three unnormalized class scores.

    Fixed 64x64 input. Explicit padding avoids PyTorch/TF SAME ambiguity.
    A 4x4 pooled grid retains finger location at a small parameter cost.
    """

    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AvgPool2d(2),
        )
        self.classifier = nn.Linear(32 * 4 * 4, num_classes)

    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))

GrayGestureCNN(
  (features): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  

## 4. 跟踪每层形状与计算量

3×3 卷积使用 padding=1，所以本层高宽不变。2×2 池化将高宽各减半。
参数量决定权重存储的一部分，MACs 表示卷积/全连接的乘加次数；两者都不能直接换算成 FPGA FPS。

In [4]:
profile = profile_model(model)
for layer in profile["layers"]:
    print(f"{layer['layer']:16s} {layer['type']:12s} {str(layer['output']):20s} MACs={layer['macs']:,}")
print(f"Parameters: {profile['parameters']:,}")
print(f"Conv/linear MACs: {profile['macs']:,}")
print(f"Raw INT8 weight-count estimate: {profile['parameters']/1024:.2f} KiB")
print("该估算不是最终 tflite 大小：还包含 int32 bias、量化参数和模型元数据。")

features.0       Conv2d       [1, 8, 64, 64]       MACs=294,912
features.1       ReLU         [1, 8, 64, 64]       MACs=0
features.2       MaxPool2d    [1, 8, 32, 32]       MACs=0
features.3       Conv2d       [1, 16, 32, 32]      MACs=1,179,648
features.4       ReLU         [1, 16, 32, 32]      MACs=0
features.5       MaxPool2d    [1, 16, 16, 16]      MACs=0
features.6       Conv2d       [1, 32, 16, 16]      MACs=1,179,648
features.7       ReLU         [1, 32, 16, 16]      MACs=0
features.8       MaxPool2d    [1, 32, 8, 8]        MACs=0
features.9       AvgPool2d    [1, 32, 4, 4]        MACs=0
classifier       Linear       [1, 3]               MACs=1,536
Parameters: 7,427
Conv/linear MACs: 2,655,744
Raw INT8 weight-count estimate: 7.25 KiB
该估算不是最终 tflite 大小：还包含 int32 bias、量化参数和模型元数据。


## 5. 前向计算与类别分数

未训练模型也能输出分数，但类别没有实用意义。`softmax` 仅把分数变成和为 1 的相对概率；不会赋予模型知识。

In [5]:
model.eval()
with torch.inference_mode():
    logits = model(x)
    probabilities = logits.softmax(dim=1)
print("logits:", logits)
print("probabilities:", probabilities)
assert tuple(logits.shape) == (1, 3)

logits: tensor([[-0.1101, -0.0214, -0.0766]])
probabilities: tensor([[0.3198, 0.3495, 0.3307]])


## 6. 一次训练更新究竟发生什么

用四个占位样本演示：清空旧梯度 → 计算 logits → 交叉熵 → 反向传播 → 参数更新。
标签是 0/1/2 的类别编号。交叉熵直接接收 logits，不提前 softmax。这只是流程检查，不是精度验证。

In [6]:
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
batch = torch.rand(4, 1, 64, 64)
labels = torch.tensor([0, 1, 2, 0])
before = model.features[0].weight.detach().clone()
optimizer.zero_grad(set_to_none=True)
loss = nn.functional.cross_entropy(model(batch), labels)
loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
optimizer.step()
print("loss:", loss.item())
print("first conv changed:", not torch.equal(before, model.features[0].weight))

loss: 1.1109707355499268
first conv changed: True


## 自测与下一章

1. 去掉一个池化层，Flatten 后的特征数会变成多少？分类头为什么也要改？
2. 为什么输出三个数，而不是一张标注框图？
3. 为什么“INT8 权重约几 KB”不能保证 15 FPS？

下一章用真实公开数据训练，重新初始化模型，不会接着使用本章占位数据更新后的权重。

### 语法与函数速查

- `nn.Module`：带有参数管理能力的网络基类。
- `nn.Sequential`：按顺序执行层；`forward` 定义整个数据流。
- `flatten(x, 1)`：保留 batch 维，从第 1 维开始展平。
- `detach().clone()`：创建不参与梯度追踪的独立快照。
- `eval()` 切换层的运行模式；`inference_mode()` 关闭推理阶段的自动求导记录。